In [1]:
#%pip install xgboost catboost optuna

In [9]:
# import libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder,FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report,accuracy_score,precision_score,recall_score,f1_score,balanced_accuracy_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna

In [3]:
# load data
train_data = pd.read_csv('train.csv')
train_data.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [5]:
target = "Irrigation_Need"

X = train_data.drop(columns=[target, "id"], errors="ignore")
y = train_data[target]

print(train_data.head())
print()
print("Target distribution:")
print(y.value_counts())
print()
print("Target proportions:")
print(y.value_counts(normalize=True))


   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0   0     Loamy     4.92          32.58            1.01   
1   1      Clay     7.08          56.61            0.44   
2   2      Clay     5.69          27.71            0.81   
3   3     Sandy     5.65          13.32            1.33   
4   4      Clay     7.96          59.14            0.38   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                     3.05          15.01     50.61       725.99   
1                     2.00          22.92     67.86       985.66   
2                     2.83          26.97     92.22      2201.70   
3                     0.87          13.32     61.57      1357.33   
4                     0.96          20.22     91.11      1538.20   

   Sunlight_Hours  ...  Crop_Type Crop_Growth_Stage  Season Irrigation_Type  \
0            5.90  ...  Sugarcane            Sowing    Zaid            Drip   
1            6.98  ...      Wheat        Vegetative  Kharif         Rainfed   

In [6]:
# Train/test split using only Kaggle train.csv
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

gnb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GaussianNB())
])

gnb_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [7]:
# Predicted probabilities
probs = gnb_model.predict_proba(X_test)
classes = gnb_model.named_steps["model"].classes_

probs_df = pd.DataFrame(probs, columns=[f"prob_{c}" for c in classes])
probs_df.head()


,prob_High,prob_Low,prob_Medium
0,1.784640e-21,0.998828,0.001172
1,2.332393e-02,0.021242,0.955434
2,6.509565e-06,0.054148,0.945846
3,4.269883e-15,0.996150,0.003850
4,5.240864e-15,0.965293,0.034707


In [10]:
# Baseline predictions: choose the class with the highest probability
baseline_preds = classes[np.argmax(probs, axis=1)]

print("Baseline classification report:")
print(classification_report(y_test, baseline_preds, zero_division=0))

print("Baseline balanced accuracy:", round(balanced_accuracy_score(y_test, baseline_preds), 4))


Baseline classification report:
              precision    recall  f1-score   support

        High       0.53      0.78      0.63      4202
         Low       0.88      0.80      0.83     73983
      Medium       0.70      0.77      0.73     47815

    accuracy                           0.78    126000
   macro avg       0.70      0.78      0.73    126000
weighted avg       0.80      0.78      0.79    126000

Baseline balanced accuracy: 0.7794


In [11]:
# Focus on the rare class
focus_class = "High"
focus_metric = "f1"

def threshold_predictions(probs, classes, focus_class, threshold):
    focus_index = np.where(classes == focus_class)[0][0]
    preds = np.empty(probs.shape[0], dtype=object)

    assign_focus = probs[:, focus_index] >= threshold
    preds[assign_focus] = focus_class

    if (~assign_focus).any():
        other_probs = probs[~assign_focus].copy()
        other_probs[:, focus_index] = -np.inf
        preds[~assign_focus] = classes[np.argmax(other_probs, axis=1)]

    return preds

threshold_rows = []

for threshold in np.arange(0.05, 0.96, 0.05):
    threshold_preds = threshold_predictions(probs, classes, focus_class, threshold)

    threshold_rows.append({
        "threshold": threshold,
        "predicted_high_count": np.sum(threshold_preds == focus_class),
        "high_precision": precision_score(y_test, threshold_preds, labels=[focus_class], average="macro", zero_division=0),
        "high_recall": recall_score(y_test, threshold_preds, labels=[focus_class], average="macro", zero_division=0),
        "high_f1": f1_score(y_test, threshold_preds, labels=[focus_class], average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_test, threshold_preds)
    })

threshold_results = pd.DataFrame(threshold_rows)

threshold_results.sort_values("high_f1", ascending=False).head(10)


,threshold,predicted_high_count,high_precision,high_recall,high_f1,balanced_accuracy
14,0.75,4002,0.716142,0.682056,0.698684,0.760840
15,0.80,3526,0.763188,0.640409,0.696429,0.749049
13,0.70,4439,0.671548,0.709424,0.689966,0.767718
12,0.65,4882,0.631299,0.733460,0.678556,0.773346
16,0.85,2936,0.819482,0.572584,0.674138,0.728567
11,0.60,5307,0.594875,0.751309,0.664003,0.776862
10,0.55,5726,0.560775,0.764160,0.646857,0.778622
9,0.50,6164,0.528066,0.774631,0.628015,0.779380
17,0.90,2277,0.882740,0.478344,0.620466,0.698987
8,0.45,6651,0.495865,0.784864,0.607758,0.779696


In [12]:
# Choose the threshold with the best F1 for the High class
best_row = threshold_results.loc[threshold_results["high_f1"].idxmax()]
chosen_threshold = best_row["threshold"]

threshold_preds = threshold_predictions(
    probs,
    classes,
    focus_class,
    chosen_threshold
)

print("Chosen class:", focus_class)
print("Chosen metric:", focus_metric)
print("Chosen threshold:", round(chosen_threshold, 2))
print()

print("Thresholded classification report:")
print(classification_report(y_test, threshold_preds, zero_division=0))

print("Thresholded balanced accuracy:", round(balanced_accuracy_score(y_test, threshold_preds), 4))


Chosen class: High
Chosen metric: f1
Chosen threshold: 0.75

Thresholded classification report:
              precision    recall  f1-score   support

        High       0.72      0.68      0.70      4202
         Low       0.88      0.80      0.83     73983
      Medium       0.70      0.80      0.75     47815

    accuracy                           0.80    126000
   macro avg       0.76      0.76      0.76    126000
weighted avg       0.80      0.80      0.80    126000

Thresholded balanced accuracy: 0.7608


In [13]:
baseline_high_f1 = f1_score(
    y_test,
    baseline_preds,
    labels=[focus_class],
    average="macro",
    zero_division=0
)

threshold_high_f1 = f1_score(
    y_test,
    threshold_preds,
    labels=[focus_class],
    average="macro",
    zero_division=0
)

baseline_high_recall = recall_score(
    y_test,
    baseline_preds,
    labels=[focus_class],
    average="macro",
    zero_division=0
)

threshold_high_recall = recall_score(
    y_test,
    threshold_preds,
    labels=[focus_class],
    average="macro",
    zero_division=0
)

summary = pd.DataFrame([
    {
        "prediction_type": "baseline argmax",
        "threshold": "default",
        "high_f1": baseline_high_f1,
        "high_recall": baseline_high_recall,
        "balanced_accuracy": balanced_accuracy_score(y_test, baseline_preds)
    },
    {
        "prediction_type": "thresholded High",
        "threshold": chosen_threshold,
        "high_f1": threshold_high_f1,
        "high_recall": threshold_high_recall,
        "balanced_accuracy": balanced_accuracy_score(y_test, threshold_preds)
    }
])

summary


,prediction_type,threshold,high_f1,high_recall,balanced_accuracy
0,baseline argmax,default,0.627493,0.775107,0.779448
1,thresholded High,0.75,0.698684,0.682056,0.760840


I selected the `High` irrigation need class because it is the rarest class and is also important from a practical standpoint: missing fields that need high irrigation could be costly. I used F1-score for the `High` class as my main metric because it balances precision and recall.

The baseline Gaussian Naive Bayes model used the default rule of assigning each row to the class with the highest predicted probability. Then I tested several probability thresholds for the `High` class and chose the threshold that gave the best `High` class F1-score on the held-out test split.

After applying the threshold, the `High` class F1-score changed from the baseline value shown in the summary table to the thresholded value shown in the summary table. The main tradeoff was that changing the threshold affected precision and recall: lowering the threshold usually predicts `High` more often, which can improve recall but may reduce precision.

Compared with stronger models such as Random Forest, XGBoost, LightGBM, or CatBoost, Gaussian Naive Bayes is much simpler and faster, but it usually performs worse because it assumes the features are conditionally independent within each class. The boosted/tree-based models can capture nonlinear relationships and interactions among soil, weather, crop, and irrigation variables more effectively.


baseline High F1 = ~0.627

thresholded High F1 = ~0.698 

chosen threshold = High class